# Falconsai T5 Bullet Specialist — Full GPU Quantization Benchmark

This notebook starts **from scratch** and benchmarks:

`JayShah07/falconai-text-bullet-t5`

on a **Google Colab NVIDIA GPU**.

It does **not use ONNX**.

## Candidates

| ID | Candidate | Precision / method |
|---|---|---|
| A | PyTorch FP32 | baseline |
| B | PyTorch FP16 | native half precision |
| C | PyTorch BF16 | only if GPU supports BF16 |
| D | bitsandbytes LLM.int8 | INT8 with outlier handling |
| E | bitsandbytes LLM.int8 + FP16 LM head | hybrid INT8 |
| F | bitsandbytes NF4 | 4-bit |
| G | Optimum Quanto INT8 | weight-only INT8 |
| H | Optimum Quanto INT4 | weight-only INT4 |
| I | TorchAO INT8 weight-only | A16W8 |
| J | TorchAO dynamic INT8 | A8W8 |
| K | TorchAO INT4 weight-only | A16W4 |
| L | TorchAO selective encoder INT8 | encoder INT8, decoder/LM head FP16 |

## What is measured

For every successful candidate:

- model load / quantization time
- model memory footprint
- peak CUDA memory
- mean / median / p95 generation latency
- tokens/sec
- ROUGE-1 / ROUGE-2 / ROUGE-L
- BERTScore precision / recall / F1
- bullet format
- predicted vs reference bullet count
- bullet-count error
- compression ratio
- exact output change vs FP32

## Why INT8 can sometimes preserve accuracy better

Naive INT8 can perturb logits enough to change greedy decoding or EOS decisions. This notebook therefore tests:

- **bitsandbytes LLM.int8()**, which preserves outlier computations at higher precision
- **weight-only INT8**, which avoids activation quantization
- **skipping the LM head**
- **selective encoder-only INT8**

Those are the main practical ways to reduce quantization error before considering QAT.

## Important hardware note

This notebook measures **GPU inference only**. GPU and CPU speed rankings are not interchangeable. A T4, for example, has Tensor Core acceleration for FP16/INT8 that ordinary CPUs do not.

In [ ]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================
#
# Run on a fresh Colab GPU runtime.
#
# torchao >= 0.15 is required for the modern config-object API.
# No ONNX / Optimum-ONNX / diffusers are needed here.
# ============================================================

!pip install -q -U     transformers     accelerate     "bitsandbytes>=0.45"     optimum-quanto     "torchao>=0.15"     sentencepiece     pandas     tqdm     rouge-score     bert-score     psutil

print("Install complete.")
print("If Colab asks for a restart, restart once and continue from Cell 2.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 18.3 MB/s eta 0:00:00
ERR

In [ ]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import os
import gc
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch

from tqdm.auto import tqdm

import transformers
import bitsandbytes as bnb

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    QuantoConfig,
)

from rouge_score import rouge_scorer
from bert_score import BERTScorer

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)

try:
    import torchao
    print("torchao:", getattr(torchao, "__version__", "installed"))
except Exception as exc:
    print("torchao import problem:", repr(exc))

torch: 2.11.0+cu128
transformers: 5.17.0
bitsandbytes: 0.50.2
torchao: 0.18.0


In [ ]:
# ============================================================
# CELL 3 — GPU CHECK
# ============================================================

assert torch.cuda.is_available(), (
    "No CUDA GPU detected. Colab: Runtime -> Change runtime type -> GPU."
)

DEVICE = torch.device("cuda:0")

props = torch.cuda.get_device_properties(0)

print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", f"{props.major}.{props.minor}")
print("VRAM GB:", round(props.total_memory / 1024**3, 2))
print("BF16 supported:", torch.cuda.is_bf16_supported())

GPU: Tesla T4
Compute capability: 7.5
VRAM GB: 14.56
BF16 supported: True


In [ ]:
# ============================================================
# CELL 4 — CONFIGURATION
# ============================================================

MODEL_ID = "JayShah07/falconai-text-bullet-t5"

# Same 500-row external evaluation CSV used in your corrected benchmark.
EVAL_CSV = "/content/output_with_bullet_points.csv"

MAX_INPUT_LENGTH = 2048
MAX_OUTPUT_LENGTH = 256
MAX_EVAL_ROWS = 500

SEED = 42
BULLET_TOKEN = "<BULLET>"

# Warmups are not included in reported latency.
NUM_WARMUPS = 3

ROOT = Path("/content/t5_gpu_quant")
RAW_DIR = ROOT / "raw"
SCORED_DIR = ROOT / "scored"
REPORT_DIR = ROOT / "reports"

for d in [ROOT, RAW_DIR, SCORED_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Model:", MODEL_ID)
print("Evaluation CSV:", EVAL_CSV)

Model: JayShah07/falconai-text-bullet-t5
Evaluation CSV: /content/output_with_bullet_points.csv


In [ ]:
# ============================================================
# CELL 5 — EXACT TASK PROMPT
# ============================================================

TASK_INSTRUCTION = '''
Convert the following English text into concise bullet points containing all materially important information.

Follow these rules:

- Extract all important and independently useful points.
- The number of bullets must depend entirely on the information in the text.
- Never use a fixed number of bullets.
- Use one bullet for each distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, metadata, boilerplate, and trivial details.
- Do not repeat the same information in multiple bullets.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add, infer, or assume information that is not supported by the source text.
- Do not turn contextual information into new advice or recommendations.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.
- Start every bullet with "- ".

Text:
'''.strip()


def build_encoder_text(text):
    return TASK_INSTRUCTION + "\n" + str(text).strip()

In [ ]:
# ============================================================
# CELL 6 — LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

bullet_id = tokenizer.convert_tokens_to_ids(BULLET_TOKEN)

print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocab size:", len(tokenizer))
print("<BULLET> ID:", bullet_id)
print("UNK ID:", tokenizer.unk_token_id)

assert bullet_id != tokenizer.unk_token_id, (
    "<BULLET> token is missing from the tokenizer."
)

config.json:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

Tokenizer: T5Tokenizer
Vocab size: 32101
<BULLET> ID: 32100
UNK ID: 2


In [ ]:
# ============================================================
# CELL 7 — LOAD / CLEAN EVALUATION CSV
# ============================================================

REQUIRED_COLUMNS = [
    "text",
    "source",
    "example_id",
    "bullet_points",
]


def clean_dataframe(df):
    df = df.copy()
    df = df.dropna(subset=["text", "bullet_points"])

    df["text"] = df["text"].astype(str).str.strip()
    df["bullet_points"] = df["bullet_points"].astype(str).str.strip()

    df = df[
        (df["text"] != "")
        &
        (df["bullet_points"] != "")
    ]

    return df.reset_index(drop=True)


if not os.path.exists(EVAL_CSV):
    raise FileNotFoundError(
        f"{EVAL_CSV} not found. Upload the evaluation CSV first."
    )

eval_df = clean_dataframe(
    pd.read_csv(EVAL_CSV)
)

for column in REQUIRED_COLUMNS:
    assert column in eval_df.columns, f"Missing required column: {column}"

eval_df = (
    eval_df
    .iloc[:MAX_EVAL_ROWS]
    .copy()
    .reset_index(drop=True)
)

print("Evaluation rows:", len(eval_df))
display(eval_df.head())

Evaluation rows: 500


,text,source,example_id,bullet_points
0,Powell: N. Korea Blast Not Nuclear Event The U...,ag_news,0,- Powell says the large North Korean explosion...
1,Jerusalem (CNN) -- Two attacks carried out aga...,cnn_dailymail,1,- Two recent attacks on Palestinians sparked I...
2,Former Kan. Junior College Coach Indicted (AP)...,ag_news,2,- Former Kansas junior college basketball coac...
3,The Ethiopian Airlines flight was travelling f...,xsum,3,- Ethiopian Airlines Flight ET500 was travelin...
4,"Minami Sanriku, Japan (CNN) -- A 60-year-old ...",cnn_dailymail,4,"- A 60‑year‑old man, Hiromitsu Shinkawa, was r..."


In [ ]:
# ============================================================
# CELL 8 — DEFINE SMOKE TEST BEFORE ANY CANDIDATE
# ============================================================
#
# This cell is intentionally BEFORE run_candidate().
# Every quantizer is tested here before we spend time on 500 rows.
# ============================================================

TEST_TEXT = '''
Acme reported quarterly revenue of $4.2 billion, up 12% year over year.

Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.

The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.

Management warned that European demand weakened in July.
'''

print(TEST_TEXT)


Acme reported quarterly revenue of $4.2 billion, up 12% year over year.

Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.

The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.

Management warned that European demand weakened in July.



In [ ]:
# ============================================================
# CELL 9 — COMMON TEXT / QUALITY HELPERS
# ============================================================

def postprocess_generated_text(raw):
    text = str(raw).strip()

    if BULLET_TOKEN in text:
        pieces = [
            piece.strip()
            for piece in text.split(BULLET_TOKEN)
            if piece.strip()
        ]

        return "\n".join(
            "- " + piece
            for piece in pieces
        )

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if lines and all(line.startswith("- ") for line in lines):
        return "\n".join(lines)

    return "- " + text if text else ""


def clean_raw_decode(raw):
    if tokenizer.pad_token:
        raw = raw.replace(tokenizer.pad_token, "")

    if tokenizer.eos_token:
        raw = raw.replace(tokenizer.eos_token, "")

    return raw.strip()


def count_bullets(text):
    return sum(
        line.strip().startswith("- ")
        for line in str(text).splitlines()
    )


def bullet_format_score(text):
    lines = [
        line.strip()
        for line in str(text).splitlines()
        if line.strip()
    ]

    if not lines:
        return 0.0

    return (
        sum(line.startswith("- ") for line in lines)
        /
        len(lines)
    )


def word_count(text):
    return len(str(text).split())


rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)

In [ ]:
# ============================================================
# CELL 10 — CUDA / MEMORY HELPERS
# ============================================================

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def model_footprint_mb(model):
    try:
        return model.get_memory_footprint() / 1024**2
    except Exception:
        total = 0

        for p in model.parameters():
            total += p.numel() * p.element_size()

        for b in model.buffers():
            total += b.numel() * b.element_size()

        return total / 1024**2


def cuda_allocated_mb():
    return torch.cuda.memory_allocated(0) / 1024**2


def cuda_peak_mb():
    return torch.cuda.max_memory_allocated(0) / 1024**2

In [ ]:
# ============================================================
# CELL 11 — CORRECT GPU GENERATION FUNCTION
# ============================================================
#
# CUDA is asynchronous, so synchronize before and after model.generate().
#
# use_cache=True enables T5 decoder KV caching.
#
# Generation is deterministic:
#   do_sample=False
#   num_beams=1
# ============================================================

def generate_gpu(model, text):

    inputs = tokenizer(
        build_encoder_text(text),
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    # Place inputs on the same CUDA device.
    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    input_tokens = int(
        inputs["input_ids"].shape[-1]
    )

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():

        ids = model.generate(
            **inputs,
            max_length=MAX_OUTPUT_LENGTH,
            do_sample=False,
            num_beams=1,
            no_repeat_ngram_size=3,
            use_cache=True,
        )

    torch.cuda.synchronize()

    latency = time.perf_counter() - start

    raw = tokenizer.decode(
        ids[0],
        skip_special_tokens=False,
    )

    raw = clean_raw_decode(raw)
    prediction = postprocess_generated_text(raw)

    output_tokens = int(
        (
            ids[0]
            !=
            tokenizer.pad_token_id
        )
        .sum()
        .item()
    )

    return {
        "output": prediction,
        "raw_output": raw,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency_seconds": latency,
        "tokens_per_second": (
            output_tokens / latency
            if latency > 0
            else np.nan
        ),
    }

In [ ]:
# ============================================================
# CELL 12 — FULL BENCHMARK LOOP
# ============================================================

def benchmark_model(name, model, dataframe):

    # Warmup outside measurements.
    print("Warmup:", name)

    for _ in range(NUM_WARMUPS):
        _ = generate_gpu(
            model,
            dataframe.iloc[0]["text"],
        )

    torch.cuda.synchronize()

    rows = []

    for _, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe),
        desc=name,
    ):

        try:

            result = generate_gpu(
                model,
                row["text"],
            )

            rows.append({
                "example_id": row["example_id"],
                "source": row["source"],
                "text": row["text"],
                "reference": row["bullet_points"],

                "prediction": result["output"],
                "raw_output": result["raw_output"],

                "input_tokens": result["input_tokens"],
                "output_tokens": result["output_tokens"],

                "latency_seconds": result["latency_seconds"],
                "tokens_per_second": result["tokens_per_second"],
            })

        except Exception as exc:

            rows.append({
                "example_id": row["example_id"],
                "source": row["source"],
                "text": row["text"],
                "reference": row["bullet_points"],
                "prediction": "",
                "error": repr(exc),
            })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# CELL 13 — CANDIDATE RUNNER
# ============================================================
#
# Each candidate:
#
# 1. Clears GPU memory
# 2. Loads / quantizes model
# 3. Runs Acme smoke test
# 4. Runs the full 500 rows
# 5. Saves raw results immediately
# 6. Records latency / throughput / memory
# 7. Deletes model before next candidate
#
# A broken quantizer therefore does not waste 500 generations.
# ============================================================

RAW_RESULTS = {}
CANDIDATE_META = []
FAILED_CANDIDATES = []


def run_candidate(name, loader_fn):

    print("\n" + "=" * 100)
    print("CANDIDATE:", name)
    print("=" * 100)

    clear_gpu()

    try:

        torch.cuda.reset_peak_memory_stats(0)

        load_start = time.perf_counter()

        model = loader_fn()

        torch.cuda.synchronize()

        load_seconds = (
            time.perf_counter()
            -
            load_start
        )

        footprint_mb = model_footprint_mb(model)

        print("Load / quantization time:", round(load_seconds, 3), "s")
        print("Model footprint:", round(footprint_mb, 2), "MB")
        print("GPU allocated:", round(cuda_allocated_mb(), 2), "MB")

        # ----------------------------------------------------
        # SMOKE TEST
        # ----------------------------------------------------

        smoke = generate_gpu(
            model,
            TEST_TEXT,
        )

        print("\nSMOKE OUTPUT:")
        print(smoke["output"])

        print(
            "\nLatency:",
            round(smoke["latency_seconds"], 4),
            "s"
        )

        print(
            "Tokens/sec:",
            round(smoke["tokens_per_second"], 2)
        )

        # Acme should produce normal bullet-form text.
        if not smoke["output"].startswith("- "):
            raise RuntimeError(
                "Smoke test produced malformed text. Candidate rejected."
            )

        # Simple corruption guard.
        if smoke["output_tokens"] >= MAX_OUTPUT_LENGTH - 2:
            print(
                "WARNING: smoke output reached almost maximum length. "
                "Inspect before trusting this candidate."
            )

        # ----------------------------------------------------
        # 500-ROW BENCHMARK
        # ----------------------------------------------------

        result_df = benchmark_model(
            name,
            model,
            eval_df,
        )

        RAW_RESULTS[name] = result_df

        raw_path = (
            RAW_DIR
            /
            f"{name}_raw.csv"
        )

        result_df.to_csv(
            raw_path,
            index=False,
        )

        latency = result_df["latency_seconds"].dropna()
        throughput = result_df["tokens_per_second"].dropna()

        meta = {
            "model": name,
            "available": True,
            "load_seconds": load_seconds,
            "model_footprint_mb": footprint_mb,
            "peak_cuda_mb": cuda_peak_mb(),

            "mean_latency_seconds": latency.mean(),
            "median_latency_seconds": latency.median(),
            "p95_latency_seconds": latency.quantile(0.95),

            "avg_tokens_per_second": throughput.mean(),
        }

        CANDIDATE_META.append(meta)

        print("\nPERFORMANCE SUMMARY")
        print("Mean latency:", round(latency.mean(), 4), "s")
        print("Median latency:", round(latency.median(), 4), "s")
        print("P95 latency:", round(latency.quantile(0.95), 4), "s")
        print("Avg tokens/sec:", round(throughput.mean(), 2))
        print("Peak CUDA MB:", round(cuda_peak_mb(), 2))
        print("Saved:", raw_path)

    except Exception as exc:

        print("\nFAILED:", name)
        print(repr(exc))

        FAILED_CANDIDATES.append({
            "model": name,
            "error": repr(exc),
        })

    finally:

        if "model" in locals():
            del model

        clear_gpu()

        print(
            "GPU allocated after cleanup:",
            round(cuda_allocated_mb(), 2),
            "MB"
        )

# A — PyTorch FP32 baseline

In [ ]:
# ============================================================
# CELL 14 — A: PYTORCH FP32
# ============================================================

def load_A():

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float32,
    )

    model = model.to(DEVICE).eval()

    return model


run_candidate(
    "A_PyTorch_FP32",
    load_A,
)


CANDIDATE: A_PyTorch_FP32


model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

Load / quantization time: 4.12 s
Model footprint: 230.76 MB
GPU allocated: 230.76 MB

SMOKE OUTPUT:
- Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
- Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.
- The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.
- Management warned that European demand weakened in July.

Latency: 2.0123 s
Tokens/sec: 39.26
Warmup: A_PyTorch_FP32


A_PyTorch_FP32:   0%|          | 0/500 [00:00<?, ?it/s]


PERFORMANCE SUMMARY
Mean latency: 0.9704 s
Median latency: 0.8947 s
P95 latency: 1.9467 s
Avg tokens/sec: 97.28
Peak CUDA MB: 682.94
Saved: /content/t5_gpu_quant/raw/A_PyTorch_FP32_raw.csv
GPU allocated after cleanup: 8.12 MB


# B — PyTorch FP16

This should generally be a much better T4 baseline than FP32.

In [ ]:
# ============================================================
# CELL 15 — B: PYTORCH FP16
# ============================================================

def load_B():

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
    )

    model = model.to(DEVICE).eval()

    return model


run_candidate(
    "B_PyTorch_FP16",
    load_B,
)


CANDIDATE: B_PyTorch_FP16


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Load / quantization time: 0.561 s
Model footprint: 139.38 MB
GPU allocated: 148.16 MB

SMOKE OUTPUT:
- Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
- Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.
- The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.
- Management warned that European demand weakened in July.

Latency: 1.079 s
Tokens/sec: 73.22
Warmup: B_PyTorch_FP16


B_PyTorch_FP16:   0%|          | 0/500 [00:00<?, ?it/s]


PERFORMANCE SUMMARY
Mean latency: 1.0329 s
Median latency: 0.961 s
P95 latency: 2.0404 s
Avg tokens/sec: 87.12
Peak CUDA MB: 530.21
Saved: /content/t5_gpu_quant/raw/B_PyTorch_FP16_raw.csv
GPU allocated after cleanup: 8.12 MB


# C — PyTorch BF16

Skipped automatically if the GPU does not report native BF16 support. T4 normally does not.

In [ ]:
# ============================================================
# CELL 16 — C: PYTORCH BF16
# ============================================================

if torch.cuda.is_bf16_supported():

    def load_C():

        model = AutoModelForSeq2SeqLM.from_pretrained(
            MODEL_ID,
            dtype=torch.bfloat16,
        )

        return model.to(DEVICE).eval()


    run_candidate(
        "C_PyTorch_BF16",
        load_C,
    )

else:

    print("Skipping BF16: GPU does not report native BF16 support.")

    FAILED_CANDIDATES.append({
        "model": "C_PyTorch_BF16",
        "error": "Native BF16 not supported by this GPU.",
    })


CANDIDATE: C_PyTorch_BF16


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Load / quantization time: 0.591 s
Model footprint: 115.38 MB
GPU allocated: 124.16 MB

SMOKE OUTPUT:
- Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
- Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.
- The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.
- Management warned that European demand weakened in July.

Latency: 1.0335 s
Tokens/sec: 76.44
Warmup: C_PyTorch_BF16


C_PyTorch_BF16:   0%|          | 0/500 [00:00<?, ?it/s]


PERFORMANCE SUMMARY
Mean latency: 1.1196 s
Median latency: 1.0643 s
P95 latency: 2.1262 s
Avg tokens/sec: 80.73
Peak CUDA MB: 504.21
Saved: /content/t5_gpu_quant/raw/C_PyTorch_BF16_raw.csv
GPU allocated after cleanup: 8.12 MB


# D — bitsandbytes LLM.int8()

`LLM.int8()` is specifically designed to preserve outlier values at higher precision instead of naively pushing all computation through INT8.

The default outlier threshold commonly used by the integration is `6.0`.

In [ ]:
# ============================================================
# CELL 17 — D: BITSANDBYTES LLM.INT8
# ============================================================

def load_D():

    qconfig = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_ID,
        quantization_config=qconfig,
        device_map={"": 0},
        dtype=torch.float16,
    )

    return model.eval()


run_candidate(
    "D_BNB_LLM_INT8",
    load_D,
)


CANDIDATE: D_BNB_LLM_INT8


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Load / quantization time: 0.611 s
Model footprint: 109.38 MB
GPU allocated: 117.74 MB

SMOKE OUTPUT:
- Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
- Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.
- The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.
- Management warned that European demand weakened in July.

Latency: 2.4346 s
Tokens/sec: 32.45
Warmup: D_BNB_LLM_INT8


D_BNB_LLM_INT8:   0%|          | 0/500 [00:00<?, ?it/s]


PERFORMANCE SUMMARY
Mean latency: 2.8577 s
Median latency: 2.6978 s
P95 latency: 5.5712 s
Avg tokens/sec: 31.59
Peak CUDA MB: 501.45
Saved: /content/t5_gpu_quant/raw/D_BNB_LLM_INT8_raw.csv
GPU allocated after cleanup: 9.12 MB


# E — bitsandbytes INT8 but LM head stays FP16

This tests whether preserving the final token projection reduces changes in bullet count / EOS behavior.

In [ ]:
# ============================================================
# CELL 18 — E: BNB INT8, SKIP LM_HEAD
# ============================================================

def load_E():

    qconfig = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_skip_modules=[
            "lm_head",
        ],
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_ID,
        quantization_config=qconfig,
        device_map={"": 0},
        dtype=torch.float16,
    )

    return model.eval()


run_candidate(
    "E_BNB_INT8_SKIP_LM_HEAD",
    load_E,
)


CANDIDATE: E_BNB_INT8_SKIP_LM_HEAD


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Load / quantization time: 0.629 s
Model footprint: 109.38 MB
GPU allocated: 118.74 MB

SMOKE OUTPUT:
- Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
- Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.
- The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.
- Management warned that European demand weakened in July.

Latency: 2.3829 s
Tokens/sec: 33.15
Warmup: E_BNB_INT8_SKIP_LM_HEAD


E_BNB_INT8_SKIP_LM_HEAD:   0%|          | 0/500 [00:00<?, ?it/s]


PERFORMANCE SUMMARY
Mean latency: 2.8462 s
Median latency: 2.722 s
P95 latency: 5.6593 s
Avg tokens/sec: 31.71
Peak CUDA MB: 501.45
Saved: /content/t5_gpu_quant/raw/E_BNB_INT8_SKIP_LM_HEAD_raw.csv
GPU allocated after cleanup: 9.12 MB


# F — bitsandbytes NF4 INT4

NF4 uses 4-bit weights with FP16 compute on T4.

In [ ]:
# ============================================================
# CELL 19 — F: BNB NF4
# ============================================================

def load_F():

    qconfig = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=False,
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_ID,
        quantization_config=qconfig,
        device_map={"": 0},
        dtype=torch.float16,
    )

    return model.eval()


run_candidate(
    "F_BNB_NF4_INT4",
    load_F,
)


CANDIDATE: F_BNB_NF4_INT4


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Load / quantization time: 0.567 s
Model footprint: 94.38 MB
GPU allocated: 106.07 MB

SMOKE OUTPUT:
- Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
- Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.
- The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.
- Management warned that European demand weakened in July.

Latency: 1.3711 s
Tokens/sec: 57.62
Warmup: F_BNB_NF4_INT4


F_BNB_NF4_INT4:   0%|          | 0/500 [00:00<?, ?it/s]


PERFORMANCE SUMMARY
Mean latency: 1.3481 s
Median latency: 1.3597 s
P95 latency: 2.4499 s
Avg tokens/sec: 68.61
Peak CUDA MB: 488.12
Saved: /content/t5_gpu_quant/raw/F_BNB_NF4_INT4_raw.csv
GPU allocated after cleanup: 9.12 MB


# G — Optimum Quanto INT8 weight-only

In [ ]:
# ============================================================
# CELL 20 — G: QUANTO INT8 WEIGHT-ONLY
# ============================================================

def load_G():

    qconfig = QuantoConfig(
        weights="int8",
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_ID,
        quantization_config=qconfig,
        device_map={"": 0},
        dtype=torch.float16,
    )

    return model.eval()


run_candidate(
    "G_QUANTO_INT8_WEIGHT_ONLY",
    load_G,
)


CANDIDATE: G_QUANTO_INT8_WEIGHT_ONLY


model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

Load / quantization time: 17.919 s
Model footprint: 139.38 MB
GPU allocated: 109.5 MB

SMOKE OUTPUT:
- Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
- Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.
- The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.
- Management warned that European demand weakened in July.

Latency: 5.2401 s
Tokens/sec: 15.08
Warmup: G_QUANTO_INT8_WEIGHT_ONLY


G_QUANTO_INT8_WEIGHT_ONLY:   0%|          | 0/500 [00:00<?, ?it/s]


PERFORMANCE SUMMARY
Mean latency: 1.6042 s
Median latency: 1.4559 s
P95 latency: 3.3416 s
Avg tokens/sec: 58.72
Peak CUDA MB: 500.32
Saved: /content/t5_gpu_quant/raw/G_QUANTO_INT8_WEIGHT_ONLY_raw.csv
GPU allocated after cleanup: 8.12 MB


# H — Optimum Quanto INT4 weight-only

In [ ]:
# ============================================================
# CELL 21 — H: QUANTO INT4 WEIGHT-ONLY
# ============================================================

def load_H():

    qconfig = QuantoConfig(
        weights="int4",
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_ID,
        quantization_config=qconfig,
        device_map={"": 0},
        dtype=torch.float16,
    )

    return model.eval()


run_candidate(
    "H_QUANTO_INT4_WEIGHT_ONLY",
    load_H,
)


CANDIDATE: H_QUANTO_INT4_WEIGHT_ONLY


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

[transformers] T5ForConditionalGeneration LOAD REPORT from: JayShah07/falconai-text-bullet-t5
Key                         | Status  | 
----------------------------+---------+-
encoder.embed_tokens.weight | MISSING | 
decoder.embed_tokens.weight | MISSING | 
lm_head.weight              | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



FAILED: H_QUANTO_INT4_WEIGHT_ONLY
RuntimeError('Error building extension \'quanto_cuda\': [1/9] /usr/local/cuda/bin/nvcc -MD -MF unpack.cuda.o.d -DTORCH_EXTENSION_NAME=quanto_cuda -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /usr/include/python3.13 -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ --expt-relaxed-constexpr -gencode=arch=compute_75,code=compute_75 -gencode=arch=compute_75,code=sm_75 --compiler-options \'-fPIC\' --expt-extended-lambda --use_fast_math -DQUANTO_CUDA_ARCH=750 -std=c++17 -c /usr/local/lib/python3.13/dist-packages/optimum/quanto/library/extensions/cuda/unpack.cu -o unpack.cuda.o \n[2/9] /usr/local/cuda/bin/nvcc -MD -MF gemm_cuda.cuda.o.d -DTORCH_EXTENSION_NAME=quanto_cuda -DTORCH_API_INCLUDE_EXTENSION_H -isyste

# TorchAO setup

TorchAO's current API uses quantization config objects. We import them once and gracefully skip the TorchAO candidates if the installed CUDA/PyTorch combination does not support them.

In [ ]:
# ============================================================
# CELL 22 — TORCHAO IMPORT
# ============================================================

TORCHAO_AVAILABLE = True
TORCHAO_ERROR = ""

try:

    from torchao.quantization import (
        quantize_,
        Int8WeightOnlyConfig,
        Int8DynamicActivationInt8WeightConfig,
        Int4WeightOnlyConfig,
    )

    print("TorchAO quantization API: OK")

except Exception as exc:

    TORCHAO_AVAILABLE = False
    TORCHAO_ERROR = repr(exc)

    print("TorchAO unavailable:", TORCHAO_ERROR)

TorchAO quantization API: OK


# I — TorchAO INT8 weight-only

In [ ]:
# ============================================================
# CELL 23 — I: TORCHAO INT8 WEIGHT-ONLY
# ============================================================

if TORCHAO_AVAILABLE:

    def load_I():

        model = AutoModelForSeq2SeqLM.from_pretrained(
            MODEL_ID,
            dtype=torch.float16,
        ).to(DEVICE).eval()

        # Model is already on CUDA. Modern quantize_ modifies
        # supported Linear modules in place.
        quantize_(
            model,
            Int8WeightOnlyConfig(),
        )

        return model


    run_candidate(
        "I_TORCHAO_INT8_WEIGHT_ONLY",
        load_I,
    )

else:

    FAILED_CANDIDATES.append({
        "model": "I_TORCHAO_INT8_WEIGHT_ONLY",
        "error": TORCHAO_ERROR,
    })


CANDIDATE: I_TORCHAO_INT8_WEIGHT_ONLY


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Load / quantization time: 2.43 s
Model footprint: 170.73 MB
GPU allocated: 98.31 MB

SMOKE OUTPUT:
- Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
- Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.
- The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.
- Management warned that European demand weakened in July.

Latency: 1.4097 s
Tokens/sec: 56.04
Warmup: I_TORCHAO_INT8_WEIGHT_ONLY


I_TORCHAO_INT8_WEIGHT_ONLY:   0%|          | 0/500 [00:00<?, ?it/s]


PERFORMANCE SUMMARY
Mean latency: 1.4774 s
Median latency: 1.4196 s
P95 latency: 2.9922 s
Avg tokens/sec: 62.36
Peak CUDA MB: 480.68
Saved: /content/t5_gpu_quant/raw/I_TORCHAO_INT8_WEIGHT_ONLY_raw.csv
GPU allocated after cleanup: 8.12 MB


# L — Selective encoder INT8, decoder + LM head FP16

This is the main candidate for **accuracy-preserving INT8** if full-model quantization changes output length or EOS decisions.

In [ ]:
# ============================================================
# CELL 26 — L: TORCHAO ENCODER-ONLY INT8
# ============================================================

if TORCHAO_AVAILABLE:

    def load_L():

        model = AutoModelForSeq2SeqLM.from_pretrained(
            MODEL_ID,
            dtype=torch.float16,
        ).to(DEVICE).eval()

        # Quantize encoder only.
        #
        # Decoder and lm_head remain FP16.
        quantize_(
            model.encoder,
            Int8WeightOnlyConfig(),
        )

        return model


    run_candidate(
        "L_TORCHAO_ENCODER_INT8_DECODER_FP16",
        load_L,
    )

else:

    FAILED_CANDIDATES.append({
        "model": "L_TORCHAO_ENCODER_INT8_DECODER_FP16",
        "error": TORCHAO_ERROR,
    })


CANDIDATE: L_TORCHAO_ENCODER_INT8_DECODER_FP16


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Load / quantization time: 1.652 s
Model footprint: 139.38 MB
GPU allocated: 229.11 MB

SMOKE OUTPUT:
- Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
- Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.
- The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.
- Management warned that European demand weakened in July.

Latency: 3.0754 s
Tokens/sec: 25.69
Warmup: L_TORCHAO_ENCODER_INT8_DECODER_FP16


L_TORCHAO_ENCODER_INT8_DECODER_FP16:   0%|          | 0/500 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# CELL 27 — CANDIDATE STATUS
# ============================================================

meta_df = pd.DataFrame(CANDIDATE_META)
failed_df = pd.DataFrame(FAILED_CANDIDATES)

print("SUCCESSFUL CANDIDATES")
display(meta_df)

print("\nFAILED / SKIPPED CANDIDATES")
display(failed_df)

SUCCESSFUL CANDIDATES


""



FAILED / SKIPPED CANDIDATES


""


In [ ]:
# ============================================================
# CELL 28 — RELOAD RAW CSVs AFTER A RESTART
# ============================================================
#
# Safe to run even if RAW_RESULTS already contains data.
# ============================================================

for path in sorted(RAW_DIR.glob("*_raw.csv")):

    name = path.name.replace("_raw.csv", "")

    if name not in RAW_RESULTS:
        RAW_RESULTS[name] = pd.read_csv(path)


print("RAW RESULTS")

for name, dataframe in RAW_RESULTS.items():
    print(name, "->", len(dataframe), "rows")

RAW RESULTS
A_PyTorch_FP32 -> 500 rows
B_PyTorch_FP16 -> 500 rows
C_PyTorch_BF16 -> 500 rows
D_BNB_LLM_INT8 -> 500 rows
E_BNB_INT8_SKIP_LM_HEAD -> 500 rows
F_BNB_NF4_INT4 -> 500 rows
G_QUANTO_INT8_WEIGHT_ONLY -> 500 rows
I_TORCHAO_INT8_WEIGHT_ONLY -> 500 rows
L_TORCHAO_ENCODER_INT8_DECODER_FP16 -> 500 rows


# Quality evaluation

BERTScore is loaded only after all generation candidates are complete. It runs on GPU but is **not included in model inference latency**.

In [ ]:
# ============================================================
# CELL 29 — BERTSCORE ON GPU
# ============================================================

clear_gpu()

bert_scorer = BERTScorer(
    model_type="distilbert-base-uncased",
    lang="en",
    device="cuda",
    rescale_with_baseline=False,
)

print("BERTScore device:", torch.cuda.get_device_name(0))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore device: Tesla T4


In [ ]:
# ============================================================
# CELL 30 — COMMON QUALITY SCORER
# ============================================================

def score_candidate(name, dataframe):

    print("\n" + "=" * 100)
    print("SCORING:", name)
    print("=" * 100)

    x = dataframe.copy()

    # --------------------------------------------------------
    # ROUGE
    # --------------------------------------------------------

    r1 = []
    r2 = []
    rl = []

    for _, row in tqdm(
        x.iterrows(),
        total=len(x),
        desc=f"ROUGE {name}",
    ):

        score = rouge.score(
            row["reference"],
            row["prediction"],
        )

        r1.append(score["rouge1"].fmeasure)
        r2.append(score["rouge2"].fmeasure)
        rl.append(score["rougeL"].fmeasure)

    x["rouge1"] = r1
    x["rouge2"] = r2
    x["rougeL"] = rl

    print(
        "ROUGE:",
        round(x["rouge1"].mean(), 4),
        round(x["rouge2"].mean(), 4),
        round(x["rougeL"].mean(), 4)
    )

    # --------------------------------------------------------
    # BERTSCORE
    # --------------------------------------------------------

    print("Starting BERTScore on GPU...")

    P, R, F1 = bert_scorer.score(
        x["prediction"].tolist(),
        x["reference"].tolist(),
        batch_size=32,
        verbose=True,
    )

    x["bertscore_precision"] = P.detach().cpu().numpy()
    x["bertscore_recall"] = R.detach().cpu().numpy()
    x["bertscore_f1"] = F1.detach().cpu().numpy()

    print(
        "BERT F1:",
        round(x["bertscore_f1"].mean(), 4)
    )

    # --------------------------------------------------------
    # BULLET STRUCTURE
    # --------------------------------------------------------

    x["bullet_format"] = x["prediction"].apply(bullet_format_score)
    x["predicted_bullets"] = x["prediction"].apply(count_bullets)
    x["reference_bullets"] = x["reference"].apply(count_bullets)

    x["bullet_count_error"] = (
        x["predicted_bullets"]
        -
        x["reference_bullets"]
    ).abs()

    # --------------------------------------------------------
    # COMPRESSION
    # --------------------------------------------------------

    x["input_words"] = x["text"].apply(word_count)
    x["prediction_words"] = x["prediction"].apply(word_count)
    x["reference_words"] = x["reference"].apply(word_count)

    denominator = x["input_words"].clip(lower=1)

    x["compression_ratio"] = (
        x["prediction_words"]
        /
        denominator
    )

    x["reference_compression_ratio"] = (
        x["reference_words"]
        /
        denominator
    )

    # --------------------------------------------------------
    # PERFORMANCE
    # --------------------------------------------------------

    latency = x["latency_seconds"].dropna()
    throughput = x["tokens_per_second"].dropna()

    summary = {

        "model": name,
        "examples": len(x),

        "rouge1": x["rouge1"].mean(),
        "rouge2": x["rouge2"].mean(),
        "rougeL": x["rougeL"].mean(),

        "bertscore_precision":
            x["bertscore_precision"].mean(),

        "bertscore_recall":
            x["bertscore_recall"].mean(),

        "bertscore_f1":
            x["bertscore_f1"].mean(),

        "bullet_format":
            x["bullet_format"].mean(),

        "avg_predicted_bullets":
            x["predicted_bullets"].mean(),

        "avg_reference_bullets":
            x["reference_bullets"].mean(),

        "mean_bullet_count_error":
            x["bullet_count_error"].mean(),

        "compression_ratio":
            x["compression_ratio"].mean(),

        "reference_compression_ratio":
            x["reference_compression_ratio"].mean(),

        "avg_latency_seconds":
            latency.mean(),

        "median_latency_seconds":
            latency.median(),

        "p95_latency_seconds":
            latency.quantile(0.95),

        "avg_output_tokens":
            x["output_tokens"].mean(),

        "avg_tokens_per_second":
            throughput.mean(),
    }

    return x, summary

In [ ]:
# ============================================================
# CELL 31 — SCORE ALL CANDIDATES
# ============================================================

SCORED_RESULTS = {}
SUMMARIES = []

for name, dataframe in RAW_RESULTS.items():

    scored, summary = score_candidate(
        name,
        dataframe,
    )

    SCORED_RESULTS[name] = scored
    SUMMARIES.append(summary)

    scored.to_csv(
        SCORED_DIR / f"{name}_scored.csv",
        index=False,
    )


summary_df = pd.DataFrame(SUMMARIES)

display(summary_df)


SCORING: A_PyTorch_FP32


ROUGE A_PyTorch_FP32:   0%|          | 0/500 [00:00<?, ?it/s]

ROUGE: 0.6041 0.533 0.5603
Starting BERTScore on GPU...
calculating scores...
computing bert embedding.


  0%|          | 0/31 [00:00<?, ?it/s]

computing greedy matching.


...

  0%|          | 0/31 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/16 [00:00<?, ?it/s]

done in 4.70 seconds, 106.49 sentences/sec
BERT F1: 0.8835


,model,examples,rouge1,rouge2,rougeL,bertscore_precision,bertscore_recall,bertscore_f1,bullet_format,avg_predicted_bullets,avg_reference_bullets,mean_bullet_count_error,compression_ratio,reference_compression_ratio,avg_latency_seconds,median_latency_seconds,p95_latency_seconds,avg_output_tokens,avg_tokens_per_second
0,A_PyTorch_FP32,500,0.604084,0.533032,0.560267,0.912775,0.856849,0.882583,1.0,3.590,4.944,1.806,0.481468,0.704085,0.970414,0.894693,1.946677,89.934,97.282112
1,B_PyTorch_FP16,500,0.604044,0.532915,0.560060,0.912761,0.856850,0.882575,1.0,3.586,4.944,1.806,0.481731,0.704085,1.032881,0.961011,2.040358,90.004,87.121042
2,C_PyTorch_BF16,500,0.606410,0.535215,0.562868,0.913170,0.857529,0.883132,1.0,3.570,4.944,1.794,0.482664,0.704085,1.119642,1.064330,2.126202,90.002,80.734919
3,D_BNB_LLM_INT8,500,0.605169,0.534890,0.562750,0.912292,0.857807,0.882906,1.0,3.570,4.944,1.822,0.482610,0.704085,2.857732,2.697755,5.571153,90.172,31.586730
4,E_BNB_INT8_SKIP_LM_HEAD,500,0.605169,0.534890,0.562750,0.912292,0.857807,0.882906,1.0,3.570,4.944,1.822,0.482610,0.704085,2.846194,2.722039,5.659266,90.172,31.705987
5,F_BNB_NF4_INT4,500,0.627288,0.557804,0.584059,0.915641,0.866157,0.889122,1.0,3.650,4.944,1.698,0.500800,0.704085,1.348059,1.359725,2.449903,92.384,68.607376
6,G_QUANTO_INT8_WEIGHT_ONLY,500,0.605376,0.534307,0.561840,0.913000,0.857539,0.883112,1.0,3.586,4.944,1.790,0.480802,0.704085,1.523618,1.481659,3.080445,90.186,59.916235
7,I_TORCHAO_INT8_WEIGHT_ONLY,500,0.610001,0.538133,0.565590,0.913011,0.859607,0.884245,1.0,3.640,4.944,1.768,0.488891,0.704085,1.421390,1.364342,2.795351,91.478,64.481764
8,L_TORCHAO_ENCODER_INT8_DECODER_FP16,500,0.606037,0.534746,0.562417,0.913325,0.857955,0.883460,1.0,3.592,4.944,1.776,0.481524,0.704085,1.067702,1.012198,2.109863,90.314,84.589073


In [ ]:
# ============================================================
# CELL 32 — MERGE MODEL MEMORY METADATA
# ============================================================

if len(CANDIDATE_META) > 0:

    meta_df = pd.DataFrame(CANDIDATE_META)

    keep = [
        "model",
        "load_seconds",
        "model_footprint_mb",
        "peak_cuda_mb",
    ]

    summary_df = summary_df.merge(
        meta_df[keep],
        on="model",
        how="left",
    )

display(summary_df)

,model,examples,rouge1,rouge2,rougeL,bertscore_precision,bertscore_recall,bertscore_f1,bullet_format,avg_predicted_bullets,avg_reference_bullets,mean_bullet_count_error,compression_ratio,reference_compression_ratio,avg_latency_seconds,median_latency_seconds,p95_latency_seconds,avg_output_tokens,avg_tokens_per_second
0,A_PyTorch_FP32,500,0.604084,0.533032,0.560267,0.912775,0.856849,0.882583,1.0,3.590,4.944,1.806,0.481468,0.704085,0.970414,0.894693,1.946677,89.934,97.282112
1,B_PyTorch_FP16,500,0.604044,0.532915,0.560060,0.912761,0.856850,0.882575,1.0,3.586,4.944,1.806,0.481731,0.704085,1.032881,0.961011,2.040358,90.004,87.121042
2,C_PyTorch_BF16,500,0.606410,0.535215,0.562868,0.913170,0.857529,0.883132,1.0,3.570,4.944,1.794,0.482664,0.704085,1.119642,1.064330,2.126202,90.002,80.734919
3,D_BNB_LLM_INT8,500,0.605169,0.534890,0.562750,0.912292,0.857807,0.882906,1.0,3.570,4.944,1.822,0.482610,0.704085,2.857732,2.697755,5.571153,90.172,31.586730
4,E_BNB_INT8_SKIP_LM_HEAD,500,0.605169,0.534890,0.562750,0.912292,0.857807,0.882906,1.0,3.570,4.944,1.822,0.482610,0.704085,2.846194,2.722039,5.659266,90.172,31.705987
5,F_BNB_NF4_INT4,500,0.627288,0.557804,0.584059,0.915641,0.866157,0.889122,1.0,3.650,4.944,1.698,0.500800,0.704085,1.348059,1.359725,2.449903,92.384,68.607376
6,G_QUANTO_INT8_WEIGHT_ONLY,500,0.605376,0.534307,0.561840,0.913000,0.857539,0.883112,1.0,3.586,4.944,1.790,0.480802,0.704085,1.523618,1.481659,3.080445,90.186,59.916235
7,I_TORCHAO_INT8_WEIGHT_ONLY,500,0.610001,0.538133,0.565590,0.913011,0.859607,0.884245,1.0,3.640,4.944,1.768,0.488891,0.704085,1.421390,1.364342,2.795351,91.478,64.481764
8,L_TORCHAO_ENCODER_INT8_DECODER_FP16,500,0.606037,0.534746,0.562417,0.913325,0.857955,0.883460,1.0,3.592,4.944,1.776,0.481524,0.704085,1.067702,1.012198,2.109863,90.314,84.589073


In [ ]:
# ============================================================
# CELL 33 — FINAL REPORT
# ============================================================

wanted = [
    "model",

    "model_footprint_mb",
    "peak_cuda_mb",
    "load_seconds",

    "rouge1",
    "rouge2",
    "rougeL",

    "bertscore_precision",
    "bertscore_recall",
    "bertscore_f1",

    "bullet_format",

    "avg_predicted_bullets",
    "avg_reference_bullets",
    "mean_bullet_count_error",

    "compression_ratio",
    "reference_compression_ratio",

    "avg_latency_seconds",
    "median_latency_seconds",
    "p95_latency_seconds",

    "avg_output_tokens",
    "avg_tokens_per_second",
]

columns = [
    c
    for c in wanted
    if c in summary_df.columns
]

report = (
    summary_df[columns]
    .sort_values("median_latency_seconds")
)

display(report)

report.to_csv(
    REPORT_DIR / "full_gpu_quantization_report.csv",
    index=False,
)

,model,rouge1,rouge2,rougeL,bertscore_precision,bertscore_recall,bertscore_f1,bullet_format,avg_predicted_bullets,avg_reference_bullets,mean_bullet_count_error,compression_ratio,reference_compression_ratio,avg_latency_seconds,median_latency_seconds,p95_latency_seconds,avg_output_tokens,avg_tokens_per_second
0,A_PyTorch_FP32,0.604084,0.533032,0.560267,0.912775,0.856849,0.882583,1.0,3.590,4.944,1.806,0.481468,0.704085,0.970414,0.894693,1.946677,89.934,97.282112
1,B_PyTorch_FP16,0.604044,0.532915,0.560060,0.912761,0.856850,0.882575,1.0,3.586,4.944,1.806,0.481731,0.704085,1.032881,0.961011,2.040358,90.004,87.121042
8,L_TORCHAO_ENCODER_INT8_DECODER_FP16,0.606037,0.534746,0.562417,0.913325,0.857955,0.883460,1.0,3.592,4.944,1.776,0.481524,0.704085,1.067702,1.012198,2.109863,90.314,84.589073
2,C_PyTorch_BF16,0.606410,0.535215,0.562868,0.913170,0.857529,0.883132,1.0,3.570,4.944,1.794,0.482664,0.704085,1.119642,1.064330,2.126202,90.002,80.734919
5,F_BNB_NF4_INT4,0.627288,0.557804,0.584059,0.915641,0.866157,0.889122,1.0,3.650,4.944,1.698,0.500800,0.704085,1.348059,1.359725,2.449903,92.384,68.607376
7,I_TORCHAO_INT8_WEIGHT_ONLY,0.610001,0.538133,0.565590,0.913011,0.859607,0.884245,1.0,3.640,4.944,1.768,0.488891,0.704085,1.421390,1.364342,2.795351,91.478,64.481764
6,G_QUANTO_INT8_WEIGHT_ONLY,0.605376,0.534307,0.561840,0.913000,0.857539,0.883112,1.0,3.586,4.944,1.790,0.480802,0.704085,1.523618,1.481659,3.080445,90.186,59.916235
3,D_BNB_LLM_INT8,0.605169,0.534890,0.562750,0.912292,0.857807,0.882906,1.0,3.570,4.944,1.822,0.482610,0.704085,2.857732,2.697755,5.571153,90.172,31.586730
4,E_BNB_INT8_SKIP_LM_HEAD,0.605169,0.534890,0.562750,0.912292,0.857807,0.882906,1.0,3.570,4.944,1.822,0.482610,0.704085,2.846194,2.722039,5.659266,90.172,31.705987


In [ ]:
# ============================================================
# CELL 34 — DELTAS VS FP32
# ============================================================

baseline = (
    report[
        report["model"]
        ==
        "A_PyTorch_FP32"
    ]
    .iloc[0]
)

rows = []

for _, candidate in report.iterrows():

    row = {

        "model":
            candidate["model"],

        "rouge1_drop_pct":
            100
            *
            (baseline["rouge1"] - candidate["rouge1"])
            /
            baseline["rouge1"],

        "rouge2_drop_pct":
            100
            *
            (baseline["rouge2"] - candidate["rouge2"])
            /
            baseline["rouge2"],

        "rougeL_drop_pct":
            100
            *
            (baseline["rougeL"] - candidate["rougeL"])
            /
            baseline["rougeL"],

        "bert_f1_drop_pct":
            100
            *
            (
                baseline["bertscore_f1"]
                -
                candidate["bertscore_f1"]
            )
            /
            baseline["bertscore_f1"],

        "median_speedup_x":
            baseline["median_latency_seconds"]
            /
            candidate["median_latency_seconds"],
    }

    if (
        "model_footprint_mb" in report.columns
        and pd.notna(candidate.get("model_footprint_mb", np.nan))
        and pd.notna(baseline.get("model_footprint_mb", np.nan))
    ):

        row["memory_reduction_pct"] = (
            100
            *
            (
                1
                -
                candidate["model_footprint_mb"]
                /
                baseline["model_footprint_mb"]
            )
        )

    rows.append(row)


delta_df = pd.DataFrame(rows)

display(delta_df)

delta_df.to_csv(
    REPORT_DIR / "quality_speed_memory_deltas.csv",
    index=False,
)

,model,rouge1_drop_pct,rouge2_drop_pct,rougeL_drop_pct,bert_f1_drop_pct,median_speedup_x
0,A_PyTorch_FP32,0.000000,0.000000,0.000000,0.000000,1.000000
1,B_PyTorch_FP16,0.006703,0.022015,0.037023,0.000898,0.930991
2,L_TORCHAO_ENCODER_INT8_DECODER_FP16,-0.323198,-0.321538,-0.383738,-0.099370,0.883911
3,C_PyTorch_BF16,-0.384958,-0.409458,-0.464165,-0.062246,0.840616
4,F_BNB_NF4_INT4,-3.841190,-4.647304,-4.246572,-0.740892,0.657996
5,I_TORCHAO_INT8_WEIGHT_ONLY,-0.979512,-0.956876,-0.950047,-0.188259,0.655769
6,G_QUANTO_INT8_WEIGHT_ONLY,-0.213796,-0.239170,-0.280689,-0.059964,0.603845
7,D_BNB_LLM_INT8,-0.179579,-0.348616,-0.443093,-0.036577,0.331643
8,E_BNB_INT8_SKIP_LM_HEAD,-0.179579,-0.348616,-0.443093,-0.036577,0.328685


In [ ]:
# ============================================================
# CELL 35 — QUALITY GATE
# ============================================================
#
# Initial deployment thresholds.
# Change them if your product requires stricter preservation.
# ============================================================

gate = delta_df.copy()

gate["quality_pass"] = (

    (gate["rougeL_drop_pct"] <= 2.0)

    &

    (gate["rouge2_drop_pct"] <= 2.0)

    &

    (gate["bert_f1_drop_pct"] <= 1.0)
)

gate["faster_than_fp32"] = (
    gate["median_speedup_x"] > 1.0
)

gate["recommended"] = (
    gate["quality_pass"]
    &
    gate["faster_than_fp32"]
)

display(
    gate.sort_values(
        ["recommended", "median_speedup_x"],
        ascending=[False, False],
    )
)

gate.to_csv(
    REPORT_DIR / "deployment_quality_gate.csv",
    index=False,
)

,model,rouge1_drop_pct,rouge2_drop_pct,rougeL_drop_pct,bert_f1_drop_pct,median_speedup_x,quality_pass,faster_than_fp32,recommended
0,A_PyTorch_FP32,0.000000,0.000000,0.000000,0.000000,1.000000,True,False,False
1,B_PyTorch_FP16,0.006703,0.022015,0.037023,0.000898,0.930991,True,False,False
2,L_TORCHAO_ENCODER_INT8_DECODER_FP16,-0.323198,-0.321538,-0.383738,-0.099370,0.883911,True,False,False
3,C_PyTorch_BF16,-0.384958,-0.409458,-0.464165,-0.062246,0.840616,True,False,False
4,F_BNB_NF4_INT4,-3.841190,-4.647304,-4.246572,-0.740892,0.657996,True,False,False
5,I_TORCHAO_INT8_WEIGHT_ONLY,-0.979512,-0.956876,-0.950047,-0.188259,0.655769,True,False,False
6,G_QUANTO_INT8_WEIGHT_ONLY,-0.213796,-0.239170,-0.280689,-0.059964,0.603845,True,False,False
7,D_BNB_LLM_INT8,-0.179579,-0.348616,-0.443093,-0.036577,0.331643,True,False,False
8,E_BNB_INT8_SKIP_LM_HEAD,-0.179579,-0.348616,-0.443093,-0.036577,0.328685,True,False,False


In [ ]:
# ============================================================
# CELL 36 — EXACT TEXT CHANGE VS FP32
# ============================================================

base_prediction = (
    RAW_RESULTS["A_PyTorch_FP32"]["prediction"]
    .reset_index(drop=True)
)

rows = []

for name, dataframe in RAW_RESULTS.items():

    candidate_prediction = (
        dataframe["prediction"]
        .reset_index(drop=True)
    )

    exact_same = (
        candidate_prediction
        ==
        base_prediction
    )

    rows.append({

        "model":
            name,

        "exact_same_as_fp32_pct":
            100 * exact_same.mean(),

        "changed_examples":
            int((~exact_same).sum()),
    })


change_df = pd.DataFrame(rows)

display(change_df)

change_df.to_csv(
    REPORT_DIR / "exact_output_change.csv",
    index=False,
)

,model,exact_same_as_fp32_pct,changed_examples
0,A_PyTorch_FP32,100.0,0
1,B_PyTorch_FP16,98.6,7
2,C_PyTorch_BF16,85.8,71
3,D_BNB_LLM_INT8,71.0,145
4,E_BNB_INT8_SKIP_LM_HEAD,71.0,145
5,F_BNB_NF4_INT4,32.8,336
6,G_QUANTO_INT8_WEIGHT_ONLY,80.2,99
7,I_TORCHAO_INT8_WEIGHT_ONLY,75.4,123
8,L_TORCHAO_ENCODER_INT8_DECODER_FP16,83.6,82


In [ ]:
# ============================================================
# CELL 37 — BULLET RECALL CHANGE VS FP32
# ============================================================
#
# This specifically detects the failure mode you saw before:
# INT8 generated fewer bullets than FP32.
# ============================================================

base_bullets = (
    RAW_RESULTS["A_PyTorch_FP32"]["prediction"]
    .apply(count_bullets)
    .reset_index(drop=True)
)

rows = []

for name, dataframe in RAW_RESULTS.items():

    candidate_bullets = (
        dataframe["prediction"]
        .apply(count_bullets)
        .reset_index(drop=True)
    )

    difference = (
        candidate_bullets
        -
        base_bullets
    )

    rows.append({

        "model":
            name,

        "avg_bullet_delta_vs_fp32":
            difference.mean(),

        "pct_examples_fewer_bullets":
            100 * (difference < 0).mean(),

        "pct_examples_same_bullets":
            100 * (difference == 0).mean(),

        "pct_examples_more_bullets":
            100 * (difference > 0).mean(),
    })


bullet_delta_df = pd.DataFrame(rows)

display(bullet_delta_df)

bullet_delta_df.to_csv(
    REPORT_DIR / "bullet_recall_change.csv",
    index=False,
)

,model,avg_bullet_delta_vs_fp32,pct_examples_fewer_bullets,pct_examples_same_bullets,pct_examples_more_bullets
0,A_PyTorch_FP32,0.000,0.0,100.0,0.0
1,B_PyTorch_FP16,-0.004,0.4,99.4,0.2
2,C_PyTorch_BF16,-0.020,3.0,94.2,2.8
3,D_BNB_LLM_INT8,-0.020,7.8,85.4,6.8
4,E_BNB_INT8_SKIP_LM_HEAD,-0.020,7.8,85.4,6.8
5,F_BNB_NF4_INT4,0.060,17.4,60.0,22.6
6,G_QUANTO_INT8_WEIGHT_ONLY,-0.004,6.0,88.0,6.0
7,I_TORCHAO_INT8_WEIGHT_ONLY,0.050,4.2,88.8,7.0
8,L_TORCHAO_ENCODER_INT8_DECODER_FP16,0.002,3.6,91.6,4.8


In [ ]:
# ============================================================
# CELL 38 — SHOW WORST QUANTIZATION REGRESSIONS
# ============================================================
#
# For each non-FP32 candidate, show examples whose ROUGE-L
# dropped most compared with FP32.
# ============================================================

fp32_scored = SCORED_RESULTS["A_PyTorch_FP32"]

for name, candidate in SCORED_RESULTS.items():

    if name == "A_PyTorch_FP32":
        continue

    comparison = pd.DataFrame({
        "text": fp32_scored["text"],
        "reference": fp32_scored["reference"],
        "fp32_prediction": fp32_scored["prediction"],
        "candidate_prediction": candidate["prediction"],
        "fp32_rougeL": fp32_scored["rougeL"],
        "candidate_rougeL": candidate["rougeL"],
    })

    comparison["rougeL_delta"] = (
        comparison["candidate_rougeL"]
        -
        comparison["fp32_rougeL"]
    )

    worst = (
        comparison
        .sort_values("rougeL_delta")
        .head(5)
    )

    print("\n" + "=" * 100)
    print("WORST REGRESSIONS:", name)
    print("=" * 100)

    display(worst)


WORST REGRESSIONS: B_PyTorch_FP16


,text,reference,fp32_prediction,candidate_prediction,fp32_rougeL,candidate_rougeL,rougeL_delta
134,The proposals for Deenethorpe Airfield in nort...,- The proposals for Deenethorpe Airfield in no...,"- The development would include between 1,000 ...","- The development would include between 1,000 ...",0.735931,0.666667,-0.069264
188,The trial has heard that Haris Mohammed attack...,- The trial has heard that Haris Mohammed atta...,- The 22-year-old died shortly after being sta...,- The 22-year-old died shortly after being sta...,0.617021,0.600000,-0.017021
495,Lyon moved back to the top of Ligue 1 for 24 h...,- Lyon moved back to the top of Ligue 1 for 24...,- Lyon moved back to the top of Ligue 1 for 24...,- Lyon moved back to the top of Ligue 1 for 24...,0.534704,0.523227,-0.011477
102,New York (CNN) -- Counsel for international ec...,"- ""We respectfully submit that the following b...",- The case has captured worldwide attention.\n...,- The case has captured worldwide attention.\n...,0.113960,0.109827,-0.004134
317,"Elected: Sage Lovell, 16, of Marietta, was ele...","- Elected: Sage Lovell, 16, of Marietta, was e...","- Sage Lovell, 16, of Marietta, was elected to...","- Sage Lovell, 16, of Marietta, was elected to...",0.303704,0.301887,-0.001817



WORST REGRESSIONS: C_PyTorch_BF16


,text,reference,fp32_prediction,candidate_prediction,fp32_rougeL,candidate_rougeL,rougeL_delta
96,By . Jason Groves and Gerri Peev . PUBLISHED: ...,- Alex Salmond was accused of producing a ‘fan...,- The Bank of England would be expected. capit...,- The Bank of England would be expected. fundi...,0.297735,0.112957,-0.184778
252,"MIke, Attached is the updated summary regardi...","- MIke, Attached is the updated summary regard...","- MIke, Attached is the updated summary regard...",- There is only one change - PGT 2003 expansio...,0.883721,0.720000,-0.163721
487,"Intel Doubles Dividend, Expands Buyback (Reute...","- Intel Doubles Dividend, Expands Buyback (Reu...","- Intel Doubles Dividend, Expands Buyback (Reu...","- Intel Doubles Dividend, Expands Buyback (Reu...",0.903226,0.785714,-0.117512
24,"(CNN)Tornadoes, fierce winds and severe thunde...","- (CNN)Tornadoes, fierce winds and severe thun...","- Severe weather is perilous anytime, of cours...","- Severe weather is perilous anytime, of cours...",0.612440,0.526786,-0.085654
409,"This is a place with fewer than 3,000 people, ...","- This is a place with fewer than 3,000 people...",- ALEX NOTT claims employment support allowanc...,- ALEX NOTT claims employment support allowanc...,0.621622,0.540146,-0.081476



WORST REGRESSIONS: D_BNB_LLM_INT8


...

,text,reference,fp32_prediction,candidate_prediction,fp32_rougeL,candidate_rougeL,rougeL_delta
451,By . Chris Paine . Well you don't see THIS eve...,- Well you don't see THIS every day.\n- A hybr...,"- A hybrid part-sheep, part-goat animal - or '...","- A hybrid part-sheep, part-goat animal - or '...",0.729282,0.369942,-0.359340
220,Enron Direkt GmbH was incorporated in Germany ...,- Enron Direkt GmbH was incorporated in German...,- Enron Direkt GmbH was incorporated in German...,- Enron Direkt GmbH was incorporated in German...,0.777778,0.550459,-0.227319
247,A father-of-three died from a burst blood vess...,- A father-of-three died from a burst blood ve...,"- Daren Hooper, 42, was discharged from hospit...",- Daren Hooper collapsed at home on January 16...,0.353698,0.140468,-0.213230
372,"Vince, 24, who is also the county's one-day ca...","- Vince, 24, who is also the county's one-day ...",- Hampshire held on for a draw against Durham ...,- Hampshire held on for a draw against Durham ...,0.671937,0.506024,-0.165913
8,"For soccer's world governing body FIFA, it is ...","- For soccer's world governing body FIFA, it i...",- Brazil faces a wave of protests about the co...,"- Brazil 2018, who are again having to field a...",0.408989,0.273349,-0.135640



WORST REGRESSIONS: I_TORCHAO_INT8_WEIGHT_ONLY


,text,reference,fp32_prediction,candidate_prediction,fp32_rougeL,candidate_rougeL,rougeL_delta
286,Macromedia Introduces Publishing Tool for EBay...,- Macromedia Introduces Publishing Tool for EB...,- Macromedia Introduces Publishing Tool for EB...,- Macromedia Introduces Publishing Tool for EBay.,0.983607,0.333333,-0.650273
220,Enron Direkt GmbH was incorporated in Germany ...,- Enron Direkt GmbH was incorporated in German...,- Enron Direkt GmbH was incorporated in German...,- Enron Direkt GmbH was incorporated in German...,0.777778,0.550459,-0.227319
46,Former Manchester United striker Michael Owen ...,- Former Manchester United striker Michael Owe...,- Michael Owen has warned Louis van Gaal that ...,- Former Manchester United striker Michael Owe...,0.549708,0.326531,-0.223177
96,By . Jason Groves and Gerri Peev . PUBLISHED: ...,- Alex Salmond was accused of producing a ‘fan...,- The Bank of England would be expected. capit...,- The Bank of England would be expected. fundi...,0.297735,0.119601,-0.178133
282,"Clement, Attached is the Enron Corp. guaranty...","- Clement, Attached is the Enron Corp. guarant...","- Clement, Attached is the Enron Corp. guarant...","- Clement, Attached is the Enron Corp.",0.363636,0.200000,-0.163636



WORST REGRESSIONS: L_TORCHAO_ENCODER_INT8_DECODER_FP16


,text,reference,fp32_prediction,candidate_prediction,fp32_rougeL,candidate_rougeL,rougeL_delta
286,Macromedia Introduces Publishing Tool for EBay...,- Macromedia Introduces Publishing Tool for EB...,- Macromedia Introduces Publishing Tool for EB...,- Macromedia Introduces Publishing Tool for EBay.,0.983607,0.333333,-0.650273
220,Enron Direkt GmbH was incorporated in Germany ...,- Enron Direkt GmbH was incorporated in German...,- Enron Direkt GmbH was incorporated in German...,- Enron Direkt GmbH was incorporated in German...,0.777778,0.550459,-0.227319
46,Former Manchester United striker Michael Owen ...,- Former Manchester United striker Michael Owe...,- Michael Owen has warned Louis van Gaal that ...,- Former Manchester United striker Michael Owe...,0.549708,0.326531,-0.223177
320,"Christian Bagley, 30, was found with wounds to...","- Christian Bagley, 30, was found with wounds ...","- Christian Bagley, 30, was found with wounds ...","- Christian Bagley, 30, was found with wounds ...",0.881356,0.730769,-0.150587
157,The competition kicked off on Friday evening a...,- The competition kicked off on Friday evening...,- Well have lost to two English National Leagu...,- Well have lost to two English National Leagu...,0.545455,0.424929,-0.120525


In [ ]:
# ============================================================
# CELL 39 — SAVE FAILED CANDIDATES + LIST REPORTS
# ============================================================

pd.DataFrame(FAILED_CANDIDATES).to_csv(
    REPORT_DIR / "failed_candidates.csv",
    index=False,
)

print("REPORT DIRECTORY:", REPORT_DIR)

for path in sorted(REPORT_DIR.glob("*")):
    print(path.name)

REPORT DIRECTORY: /content/t5_gpu_quant/reports
bullet_recall_change.csv
deployment_quality_gate.csv
exact_output_change.csv
failed_candidates.csv
full_gpu_quantization_report.csv
quality_speed_memory_deltas.csv


# What to do after this notebook

First choose the candidate that gives the best combination of:

1. **ROUGE-L / ROUGE-2 preservation**
2. **BERTScore F1 preservation**
3. **bullet-count recall**
4. **GPU memory reduction**
5. **median + p95 latency**

If full INT8 loses recall but `E_BNB_INT8_SKIP_LM_HEAD` or `L_TORCHAO_ENCODER_INT8_DECODER_FP16` preserves it, use the hybrid representation.

If no post-training INT8 method preserves enough quality, the next step is **quantization-aware training (QAT)** rather than changing the serving runtime again.

TorchAO currently documents QAT workflows, and Quanto's underlying library also supports calibration/QAT. That is the route to pursue if you want an INT8/INT4 representation while training the model to compensate for quantization noise.